# 完整文本预测与评测

复用根目录 `main.py` 的对齐、滑窗和解码。可以从项目根目录或 `notebooks/` 启动，默认使用已复核的新版基线评测官方dev。输入原文/人工答案与模型预测分开，不能将历史预测当作gold。

输出文件已存在时拒绝覆盖。官网公开 `PKU-SEGPOS/test_public.jsonline` 没有答案，预测时将 `gold_path=None`；正式训练和调参结束后才使用prepared_v3/test.jsonline作最终验收。


In [ ]:
from pathlib import Path
import os
import sys

# Notebook可能从子目录启动，统一定位项目根目录后再导入模块。
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "train.py").is_file() and (p / "main.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("请在PKU-SEGPOS项目目录或notebooks子目录中启动Notebook")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 导入业务模块
import json
from main import main as predict_main
from eval import evaluate_data, load_file


In [ ]:
model_path = "outputs/bert-prepared-v3-lr3e5-seed42/best_model"
input_path = "data/prepared_v3/dev.jsonline"
gold_path = input_path
output_path = "outputs/predictions/dev_predictions.jsonline"
# 若评测diagnostic_dev，修改input_path；gold_path保持指向同一份人工答案。
# 官网公开数据无答案，将gold_path改为None，只执行预测。


In [ ]:
predict_main(["--model", model_path, "--input", input_path, "--output", output_path])


In [ ]:
report = None
if gold_path is None:
    print("输入没有gold，本次仅生成预测，不计算F1。")
else:
    catalog = json.loads(Path("data/prepared_v3/label_catalog.json").read_text(encoding="utf-8"))
    report = evaluate_data(load_file(gold_path), load_file(output_path), set(catalog["observed_tags"]))
    print({key: report[key] for key in ("precision", "recall", "f1", "character_coverage")})
    print(report["per_tag"])


## 人工输入句子，检查字符与完整词属性

这一节可独立运行，无需先执行上方批量预测与评测。先运行一次“加载模型”和“定义函数”，再重复运行输入单元格。模型从本地加载，不写入数据或预测文件。

两种输出分别为 `字(词性)` 与 `完整词(词性)`；词边界来自模型的 BIO 解码。可打开 `show_bio` 查看 `字(B-/I-词性)`，帮助排查边界。`O` 表示未标注，仍显示对应字符。

默认使用 Viterbi；可把 `inspection_decoder` 改为 `argmax` 做对照。这是模型预测，不是人工答案。保持原文，输入不含空格、换行或其他会被 BERT tokenizer 丢弃的字符；遇到这类字符会明确报错。长句复用滑窗，不截断。


In [ ]:
from pathlib import Path
import os
import sys

# Notebook可能从子目录启动，统一定位项目根目录后再导入模块。
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "train.py").is_file() and (p / "main.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("请在PKU-SEGPOS项目目录或notebooks子目录中启动Notebook")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 导入业务模块
import torch
from transformers import AutoTokenizer, BertForTokenClassification
from main import predict_sentence

inspection_model_path = "outputs/bert-prepared-v3-lr3e5-seed42/best_model"
inspection_decoder = "viterbi"  # 也可改为 "argmax"
inspection_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inspection_tokenizer = AutoTokenizer.from_pretrained(
    inspection_model_path, use_fast=True, local_files_only=True
)
inspection_model = BertForTokenClassification.from_pretrained(
    inspection_model_path, local_files_only=True
).to(inspection_device)
inspection_model.eval()
print(f"模型已加载：{inspection_model_path}；设备：{inspection_device}")


In [ ]:
def inspect_sentence(sentence, decoder=None, show_bio=False):
    result = predict_sentence(
        sentence, inspection_model, inspection_tokenizer, inspection_device,
        decoder=decoder or inspection_decoder,
    )
    print("原句：", result["text"])
    print("字(属性)：", result["character_output"])
    print("完整词(属性)：", result["word_output"])
    if show_bio:
        print("字(BIO属性)：", result["bio_output"])
    print(f"解码：{result['decoder']}；窗口：{result['windows']}；"
          f"已标注字符覆盖率：{result['character_coverage']:.2%}；"
          f"非法BIO转移：{result['illegal_bio_transitions']}")
    return result


In [ ]:
manual_sentence = input("请输入要排查的句子：")
inspection_result = inspect_sentence(manual_sentence, show_bio=False)
# 也可直接调用：inspection_result = inspect_sentence("蜘蛛结了一张网。", show_bio=True)
# 对照：argmax_result = inspect_sentence(manual_sentence, decoder="argmax")


## 普通文本文件预测与中文词性导出

先运行上方“加载模型”单元格，再运行本节；无需运行人工输入或有标签数据评测。修改输入路径、编码和输出目录即可。默认读取 UTF-8（兼容 BOM）；GBK/GB18030 文本可显式指定 `gb18030`，解码失败不会自动猜测编码。

- `predictions.jsonline`：项目标准 `tokens + span_list`，每个非空物理行一条记录；`line_number` 是原文件行号（从 1 开始），字符坐标从 0 开始且 end 包含末端。tokens 保留该行全部空格与制表符，不含行尾换行；真正的空行不生成记录。
- `words_zh.txt`：`蜘蛛(名词) 北京(地名)` 形式的可读预测，保留原文件的缩进、空白、空行和换行方式；词之间增加空格便于阅读，O 显示为“未标注”。

每行按空白分段后调用同一个已加载模型，长段使用滑窗，不截断。空白只是分隔符，不参与词性预测，span 坐标仍对应原始行；其他被 tokenizer 丢弃的字符会报错并显示行号。由于按行和空白分段，上下文不会跨越这些边界。带空白的 JSONL 不应直接作为要求完整标注覆盖的训练数据。输出目录必须尚不存在，失败时清理本次未完成的输出，不覆盖原文或已有结果。


In [ ]:
from pathlib import Path
import json
from main import predict_text_file

# 相对路径以项目根目录为准，也可填写绝对路径。
input_text_path = Path("inputs/example.txt")  # 修改为你自己的普通文本文件
text_output_dir = Path("outputs/predictions/text_file_run")  # 再次运行时换一个新目录
text_encoding = "utf-8-sig"  # GBK/GB18030 文件可改为 "gb18030"


In [ ]:
text_prediction_summary = predict_text_file(
    input_text_path, text_output_dir,
    inspection_model, inspection_tokenizer, inspection_device,
    encoding=text_encoding, decoder=inspection_decoder, batch_size=16,
)
print(json.dumps(text_prediction_summary, ensure_ascii=False, indent=2))
print("\n中文词性输出预览（前 3 行，每行最多 500 字符）：")
with Path(text_prediction_summary["readable_output"]).open(encoding="utf-8") as stream:
    for _, line in zip(range(3), stream):
        print(line.rstrip("\r\n")[:500])
